# ch08 进阶与基线（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv` → A 台区 8760 h；窗口 72、步长 24、起点从 168 开始、按时间 8:2 切分（与 ch07 完全一致）。
>
> **每题都跟了 `# 提示：`**，照着提示能独立做完。
> 涉及 torch / LightGBM 训练的断言都是**区间或相对大小**（不是死值）；
> **朴素基线与感受野公式是确定性的**，那几个数才卡死。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 朴素基线家族（last / sn24 / sn168 / ma24） | 7 | §8.1 |
| 2 | 20 维特征 + LightGBM 24 个模型 | 11 | §8.2 |
| 3 | 特征重要性 top-10 | 1 | §8.2 |
| 4 | 与 sklearn HistGB 同族对照 | 2 | §8.3 |
| 5 | 因果卷积 + 残差块 + 感受野公式与实测 | 6 | §8.4 |
| 6 | TCN 训练 | 2 | §8.4.4 |
| 7 | ProbSparse：u、稀疏度、top-u、打分次数、蒸馏 | 4 | §8.5 |
| 8 | Transformer 训练（全注意力 vs ProbSparse） | 2 | §8.5.1 |
| 9 | 全基线对比表 | 1 | §8.6 |

In [ ]:
import math
import os
import time
from pathlib import Path

# 本题要同时用 lightgbm 与 torch，而这两个库各自静态链接了一份 OpenMP 运行时
# （lightgbm 走 libomp，torch 走 libiomp）。在 macOS/arm64 上让两份运行时各自
# 开满线程，会撞出 `OMP: Error #179: Function pthread_mutex_init failed`，
# 内核直接段错误。**在 import 之前**把线程数限成 1 就能绕开这个初始化冲突。
# 这是一条真实的工程踩坑记录，不是可有可无的仪式。
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
import torch
import torch.nn as nn
import torch.nn.functional as F

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"

pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 60)

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
print("原始行数", len(raw), "| 列", list(raw.columns))
print("lightgbm", lgb.__version__, "| torch", torch.__version__, "| CUDA 可用？", torch.cuda.is_available())

In [ ]:
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
frame = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳").sort_index()


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值（ch01 §1.3.2）。"""
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


s = clean_series(frame["负荷值"].asfreq("h"))
ser = s.values
temp = frame["温度"].asfreq("h").interpolate()

print("A 台区 %d 个整点 | 清洗后 mean %.6f" % (len(ser), ser.mean()))
print("温度 mean %.4f | 缺失 %d" % (temp.mean(), int(temp.isna().sum())))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
WINDOW, HORIZON = 72, 24
START = 168          # 起点从 168 开始：lag168 需要 168 小时历史（ch07 用同一口径）

HOLIDAY_RANGES = [
    ("2025-01-01", "2025-01-01"),   # 元旦
    ("2025-01-28", "2025-02-04"),   # 春节
    ("2025-04-04", "2025-04-06"),   # 清明
    ("2025-05-01", "2025-05-05"),   # 劳动节
    ("2025-05-31", "2025-06-02"),   # 端午
    ("2025-10-01", "2025-10-08"),   # 中秋 + 国庆
]
HOLIDAYS = set()
for _lo, _hi in HOLIDAY_RANGES:
    HOLIDAYS |= set(pd.date_range(_lo, _hi, freq="D").strftime("%Y-%m-%d"))


def mae_all(y_true, y_pred):
    """24 步合起来的单个 MAE —— 本章所有对比表的排序依据。"""
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def mae_by_step(y_true, y_pred):
    """返回长度 HORIZON 的逐步 MAE。"""
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), axis=0)

## 任务 1：朴素基线家族 —— 先给及格线定价（§8.1）

In [ ]:
origins = np.arange(START, len(ser) - HORIZON + 1)

Y = np.stack([ser[t:t + HORIZON] for t in origins])

X = np.stack([ser[t - WINDOW:t] for t in origins])

cut = int(len(origins) * 0.8)

X_test, Y_test = X[cut:], Y[cut:]
oc = origins[cut:]
t_test = frame.index[oc]
print("样本 %d | 切分点 %d | 训练 %d / 测试 %d" % (len(origins), cut, cut, len(oc)))
print("测试段 %s ~ %s" % (t_test[0], t_test[-1]))

P_last = np.tile(ser[oc - 1][:, None], (1, HORIZON))

P_sn24 = np.stack([ser[t + np.arange(1, HORIZON + 1) - 24] for t in oc])

P_sn168 = np.stack([ser[t + np.arange(1, HORIZON + 1) - 168] for t in oc])

P_ma24 = np.tile(np.array([ser[t - 24:t].mean() for t in oc])[:, None], (1, HORIZON))

BASELINES = [("last value", P_last), ("seasonal naive t-24", P_sn24),
             ("seasonal naive t-168", P_sn168), ("moving average 24", P_ma24)]
print()
print("%-22s %10s %10s %10s" % ("基线", "24步 MAE", "h=1 MAE", "h=24 MAE"))
for nm, P in BASELINES:
    print("%-22s %10.4f %10.4f %10.4f" % (nm, mae_all(Y_test, P), mae_by_step(Y_test, P)[0],
                                          mae_by_step(Y_test, P)[23]))
print()
print("及格线 = 最好的朴素基线 %.6f（%s）"
      % (min(mae_all(Y_test, P) for _, P in BASELINES),
         min(BASELINES, key=lambda kv: mae_all(Y_test, kv[1]))[0]))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
axes[0].bar([nm for nm, _ in BASELINES], [mae_all(Y_test, P) for _, P in BASELINES],
            color=["tab:gray", "tab:blue", "tab:green", "tab:orange"])
axes[0].set_ylabel("24 步 MAE (kW)")
axes[0].set_title("朴素基线家族：及格线之争")
axes[0].tick_params(axis="x", rotation=18)
for nm, P in BASELINES:
    axes[1].plot(np.arange(1, HORIZON + 1), mae_by_step(Y_test, P), marker="o", ms=3, label=nm)
axes[1].set_xlabel("预测步长 h（小时）")
axes[1].set_ylabel("MAE (kW)")
axes[1].set_title("朴素基线的误差随步长几乎平")
axes[1].legend(fontsize=8)
plt.tight_layout()

# ---- 验收 ----
assert len(origins) == 8569 and cut == 6855
assert X.shape == (8569, WINDOW) and X_test.shape == (1714, WINDOW)
assert X[0, -1] == ser[START - 1] and Y[0, 0] == ser[START], "窗口不含 t，目标是 t 起"
assert abs(mae_all(Y_test, P_last) - 60.524392) < 1e-5
assert abs(mae_all(Y_test, P_sn24) - 32.998794) < 1e-5
assert abs(mae_all(Y_test, P_sn168) - 31.560227) < 1e-5
assert abs(mae_all(Y_test, P_ma24) - 45.527850) < 1e-5
assert abs(mae_by_step(Y_test, P_last)[0] - 30.0939) < 1e-4
assert abs(mae_by_step(Y_test, P_last)[23] - 21.6066) < 1e-4
assert abs(mae_by_step(Y_test, P_sn168)[0] - 31.5845) < 1e-4
assert mae_all(Y_test, P_sn168) < mae_all(Y_test, P_sn24) < mae_all(Y_test, P_ma24) < mae_all(Y_test, P_last)

## 任务 2：特征工程 + LightGBM 真基线（§8.2）

In [ ]:
idx = pd.DatetimeIndex(frame.index[origins])      # 注意是**全部**起点，不是测试段
Ft = pd.DataFrame(index=np.arange(len(origins)))

for lag in (1, 2, 3, 24, 25, 168):
    Ft["lag%d" % lag] = ser[origins - lag]

W24 = np.stack([ser[t - 24:t] for t in origins])
Ft["ma24"], Ft["std24"] = W24.mean(1), W24.std(1)
Ft["min24"], Ft["max24"] = W24.min(1), W24.max(1)

Ft["ma168"] = np.stack([ser[t - 168:t] for t in origins]).mean(1)

for nm, period, val in (("hour", 24, idx.hour), ("dow", 7, idx.dayofweek),
                        ("month", 12, idx.month - 1)):
    Ft[nm + "_sin"] = np.sin(2 * np.pi * val / period)
    Ft[nm + "_cos"] = np.cos(2 * np.pi * val / period)

Ft["is_holiday"] = idx.strftime("%Y-%m-%d").isin(HOLIDAYS).astype(int)

T_prev = temp.values[origins - 1]
Ft["temp"] = T_prev
Ft["temp_dev"] = np.abs(T_prev - 22)

FEATS = list(Ft.columns)
Fv = np.ascontiguousarray(Ft.values, dtype="float32")
Yv = np.ascontiguousarray(Y, dtype="float32")
vc = int(cut * 0.85)                    # 训练段内部再切 15% 当验证段（按时间，不 shuffle）
F_fit, F_val, F_te = Fv[:vc].copy(), Fv[vc:cut].copy(), Fv[cut:].copy()
Y_fit, Y_val = Yv[:vc].copy(), Yv[vc:cut].copy()
print("特征 %d 个：%s" % (len(FEATS), FEATS))
print("NaN 个数 %d | 拟合段 %d / 验证段 %d / 测试段 %d"
      % (int(np.isnan(Fv).sum()), len(F_fit), len(F_val), len(F_te)))

t0 = time.time()
P_lgb = np.zeros_like(Y_test, dtype=float)
importance = np.zeros(len(FEATS))
best_iters = []
for h in range(HORIZON):
    model_h = LGBMRegressor(n_estimators=150, learning_rate=0.05, num_leaves=31,
                            subsample=0.8, colsample_bytree=0.8, random_state=42,
                            verbose=-1, n_jobs=1)
    model_h.fit(F_fit, Y_fit[:, h], eval_X=F_val, eval_y=Y_val[:, h],
                callbacks=[lgb.early_stopping(20, verbose=False)])
    P_lgb[:, h] = model_h.predict(F_te)
    importance = importance + model_h.feature_importances_
    best_iters.append(model_h.best_iteration_ or model_h.n_estimators)

print()
print("LightGBM 24 个模型共耗时 %.1f s | 平均 best_iteration %.1f（上限 150）"
      % (time.time() - t0, float(np.mean(best_iters))))
print("LightGBM 24 步 MAE %.6f（h=1 %.4f → h=24 %.4f）"
      % (mae_all(Y_test, P_lgb), mae_by_step(Y_test, P_lgb)[0], mae_by_step(Y_test, P_lgb)[23]))
print("比最好的朴素基线 %.6f 好了 %.6f（降 %.1f%%）"
      % (mae_all(Y_test, P_sn168), mae_all(Y_test, P_sn168) - mae_all(Y_test, P_lgb),
         100 * (1 - mae_all(Y_test, P_lgb) / mae_all(Y_test, P_sn168))))

# ---- 验收 ----
assert len(FEATS) == 20 and Fv.shape == (8569, 20)
assert int(np.isnan(Fv).sum()) == 0, "所有特征都必须能算出来（lag168 靠 START=168 保证）"
assert 10 < mae_all(Y_test, P_lgb) < 25, "训练结果依赖随机种子，用区间断言；但必须远好于基线"
assert mae_all(Y_test, P_lgb) < mae_all(Y_test, P_sn168) * 0.6, "至少要比基线好 40%"
assert all(1 <= it <= 150 for it in best_iters)
assert float(importance.sum()) > 0

## 任务 3：特征重要性 top-10（§8.2.1）

In [ ]:
order = np.argsort(-importance)[:10]

top_df = pd.DataFrame({"特征": [FEATS[i] for i in order],
                       "累计重要性": [int(importance[i]) for i in order]})
print(top_df.to_string(index=False))
print()
print("重要性前 3：%s" % " / ".join(FEATS[i] for i in order[:3]))
print("lag24 的排名 %d | lag168 的排名 %d | temp_dev 的排名 %d"
      % (list(order).index(FEATS.index("lag24")) + 1 if FEATS.index("lag24") in order else -1,
         list(order).index(FEATS.index("lag168")) + 1 if FEATS.index("lag168") in order else -1,
         list(order).index(FEATS.index("temp_dev")) + 1 if FEATS.index("temp_dev") in order else -1))

fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.barh([FEATS[i] for i in order][::-1], [importance[i] for i in order][::-1], color="tab:blue")
ax.set_xlabel("累计重要性（24 个模型相加，`feature_importances_`）")
ax.set_title("LightGBM 特征重要性 top-10")
plt.tight_layout()

# ---- 验收 ----
assert len(order) == 10 and len(set(order.tolist())) == 10
assert set(order.tolist()) <= set(range(len(FEATS)))
assert importance[order[0]] >= importance[order[9]] >= 0, "排序必须真的降序"

## 任务 4：同族对照 —— sklearn HistGradientBoosting（§8.3）

In [ ]:
H_SEL = (0, 11, 23)               # h=1 / h=12 / h=24 三个代表步长
LGB_MAE, HGB_MAE = {}, {}
for h in H_SEL:
    hgb = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_leaf_nodes=31,
                                        random_state=42, early_stopping=False)
    hgb.fit(F_fit, Y_fit[:, h])
    HGB_MAE[h] = float(np.mean(np.abs(hgb.predict(F_te) - Y_test[:, h])))
    LGB_MAE[h] = float(mae_by_step(Y_test, P_lgb)[h])

print("%-8s %12s %12s %10s" % ("步长", "LightGBM", "HistGB", "差值"))
for h in H_SEL:
    print("h=%-5d %12.4f %12.4f %+10.4f" % (h + 1, LGB_MAE[h], HGB_MAE[h], HGB_MAE[h] - LGB_MAE[h]))
print()
print("三点平均：LightGBM %.4f | HistGB %.4f | 差 %+.4f（相对 %.2f%%）"
      % (np.mean(list(LGB_MAE.values())), np.mean(list(HGB_MAE.values())),
         np.mean(list(HGB_MAE.values())) - np.mean(list(LGB_MAE.values())),
         100 * (np.mean(list(HGB_MAE.values())) / np.mean(list(LGB_MAE.values())) - 1)))

# ---- 验收 ----
assert set(H_SEL) == {0, 11, 23} and len(LGB_MAE) == len(HGB_MAE) == 3
assert all(10 < v < 25 for v in HGB_MAE.values()), "同族对照的两个库应在同一个量级"
for h in H_SEL:
    assert abs(HGB_MAE[h] - LGB_MAE[h]) < 4, "同族模型的差距不应该超过 4 kW"

## 任务 5：TCN 的因果卷积、残差块与感受野（§8.4）

In [ ]:
DS = (1, 2, 4, 8)               # 每个残差块的膨胀率
K = 3                          # 卷积核宽度
CH = 16                        # 通道数


class CausalConv1d(nn.Module):
    """因果卷积：只在**左边** padding，于是输出第 i 个位置只依赖输入 ≤ i 的位置。"""

    def __init__(self, cin, cout, k, d=1):
        super().__init__()
        self.pad = (k - 1) * d
        self.conv = nn.Conv1d(cin, cout, k, dilation=d)

    def forward(self, x):
        return self.conv(F.pad(x, (self.pad, 0)))      # (左, 右)


class TCNBlock(nn.Module):
    """残差块：两个同膨胀率的因果卷积 + 一条恒等跳连。"""

    def __init__(self, ch, k, d):
        super().__init__()
        self.conv1 = CausalConv1d(ch, ch, k, d)
        self.conv2 = CausalConv1d(ch, ch, k, d)

    def forward(self, x):
        return torch.relu(x + torch.relu(self.conv2(torch.relu(self.conv1(x)))))


class TCN(nn.Module):
    def __init__(self, ch=CH, k=K, ds=DS, out=HORIZON):
        super().__init__()
        self.inp = CausalConv1d(1, ch, k, 1)
        self.blocks = nn.ModuleList([TCNBlock(ch, k, d) for d in ds])
        self.head = nn.Linear(ch, out)

    def forward(self, x):
        h = self.inp(x)
        for blk in self.blocks:
            h = blk(h)
        return self.head(h[:, :, -1])


RF = 1 + (K - 1) * sum(2 * d for d in DS)

with torch.random.fork_rng():
    torch.manual_seed(0)
    # 用**纯线性**的因果卷积堆量感受野：加了 ReLU 扰动会被吃掉，量不准
    stack = nn.ModuleList()
    cin = 1
    for d in DS:
        for _ in range(2):
            stack.append(CausalConv1d(cin, 4, K, d))
            cin = 4
    clean_in = torch.zeros(1, 1, WINDOW)
    bumped = clean_in.clone()
    bumped[0, 0, 0] = 1.0
    with torch.no_grad():
        out_clean, out_bumped = clean_in, bumped
        for layer in stack:
            out_clean, out_bumped = layer(out_clean), layer(out_bumped)
    diff = (out_clean - out_bumped).abs().max(dim=1)[0][0].numpy()
    rf_measured = int((diff > 1e-8).sum())

print("感受野公式 RF = 1 + (k-1)·Σ2d = 1 + %d×%d = %d" % (K - 1, sum(2 * d for d in DS), RF))
print("实测感受野 = %d（扰动输入第 0 格，输出里有 %d 格跟着变）" % (rf_measured, rf_measured))
print("窗口长度 %d > RF %d：窗口最前面 %d 格对最后一个输出**没有影响**" % (WINDOW, RF, WINDOW - RF))
print("参数与层数：%d 个因果卷积层（含输入层），通道 %d" % (1 + 2 * len(DS), CH))

# ---- 验收 ----
assert RF == 61, "1 + (3-1)·2·(1+2+4+8) = 61"
assert rf_measured == RF, "实测感受野必须与公式一致"
assert rf_measured < WINDOW, "RF < 窗口：最前面 11 格是白给的"
assert (1 + 2 * len(DS)) == 9

## 任务 6：训练 TCN 并评估（§8.4.4）

In [ ]:
STRIDE, EPOCHS, BS, LR = 4, 12, 128, 1e-2
MX, SX = float(X[:cut].mean()), float(X[:cut].std())
MY, SY = float(Y[:cut].mean()), float(Y[:cut].std())
Xt = torch.tensor((X[:cut] - MX) / SX, dtype=torch.float32)[:, None, :]
Xe = torch.tensor((X_test - MX) / SX, dtype=torch.float32)[:, None, :]
Yt = torch.tensor((Y[:cut] - MY) / SY, dtype=torch.float32)
Xs, Ys = Xt[::STRIDE], Yt[::STRIDE]
print("深度模型训练样本 %d（stride=%d，只用 1/%d 的训练样本以守住算力预算）"
      % (len(Xs), STRIDE, STRIDE))
print("标准化只来自训练段：MX %.4f / SX %.4f | MY %.4f / SY %.4f" % (MX, SX, MY, SY))

t0 = time.time()
with torch.random.fork_rng():
    torch.manual_seed(1)
    tcn = TCN()
    opt = torch.optim.Adam(tcn.parameters(), lr=LR)
    for ep in range(EPOCHS):
        perm = torch.randperm(len(Xs))
        for i in range(0, len(Xs), BS):
            idx = perm[i:i + BS]
            opt.zero_grad()
            ((tcn(Xs[idx]) - Ys[idx]) ** 2).mean().backward()
            opt.step()
    tcn.eval()
    with torch.no_grad():
        P_tcn = tcn(Xe).numpy() * SY + MY

tcn_secs = time.time() - t0

print("TCN 训练 %.1f s（%d 个样本 × %d 轮 = %.1f 万次「样本-轮」）"
      % (tcn_secs, len(Xs), EPOCHS, len(Xs) * EPOCHS / 1e4))
print("算力效率：%.0f µs / 样本-轮（卷积在 CPU 上就是这个价）"
      % (1e6 * tcn_secs / (len(Xs) * EPOCHS)))
print("TCN 24 步 MAE %.6f（h=1 %.4f → h=24 %.4f）"
      % (mae_all(Y_test, P_tcn), mae_by_step(Y_test, P_tcn)[0], mae_by_step(Y_test, P_tcn)[23]))
print("对照：LightGBM %.6f | 最好朴素基线 %.6f"
      % (mae_all(Y_test, P_lgb), mae_all(Y_test, P_sn168)))
print("TCN 参数个数 %d" % sum(p.numel() for p in tcn.parameters()))

# ---- 验收 ----
assert P_tcn.shape == Y_test.shape == (1714, HORIZON)
assert 10 < mae_all(Y_test, P_tcn) < 60, "随机训练结果 → 区间断言"
assert mae_all(Y_test, P_tcn) > mae_all(Y_test, P_lgb), "本章结论：这个数据规模上深度模型打不过 LightGBM"
assert sum(p.numel() for p in tcn.parameters()) < 10000, "TCN 很小（省算力）"

## 任务 7：ProbSparse 注意力与蒸馏（§8.5）

In [ ]:
L = WINDOW
C_SPARSE = 5.0


class ProbSparseAttention(nn.Module):
    """Informer 的 ProbSparse 简化版：只对 top-u 个 query 做注意力。

    其余 query 的注意力输出直接取 V 的均值（等价于「均匀看所有时刻」）。
    """

    def __init__(self, d_model, c=C_SPARSE):
        super().__init__()
        self.q, self.k, self.v = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.d, self.c = d_model, c

    def forward(self, x):
        B, length, D = x.shape
        Q, K, V = self.q(x), self.k(x), self.v(x)
        U = max(1, int(math.ceil(self.c * math.log(length))))
        ks = torch.randperm(length)[:U]
        s = Q @ K[:, ks].transpose(1, 2) / math.sqrt(D)
        score = s.max(-1).values - s.mean(-1)
        u_idx = torch.topk(score, U, dim=1).indices
        full = Q @ K.transpose(1, 2) / math.sqrt(D)
        row = torch.zeros_like(full)
        row.scatter_(1, u_idx.unsqueeze(-1).expand(-1, -1, length), 1.0)
        att = torch.softmax(full, -1) * row
        att = att / att.sum(-1, keepdim=True).clamp_min(1e-9)
        out = att @ V
        mean_v = V.mean(1, keepdim=True).expand(-1, length, -1)
        return torch.where(row.sum(-1, keepdim=True) > 0, out, mean_v)


U72 = max(1, int(math.ceil(C_SPARSE * math.log(L))))
score_full = L * L
score_sparse = 2 * L * U72
print("L = %d | u = c·ln L = %.4f → 向上取整 %d（占 L 的 %.1f%%）"
      % (L, C_SPARSE * math.log(L), U72, 100 * U72 / L))
print("打分次数：全注意力 %d | ProbSparse %d | 省下 %.1f%%"
      % (score_full, score_sparse, 100 * (1 - score_sparse / score_full)))
print()
print("省下的比例随 L 变化（c = %.1f，u = ⌈c·ln L⌉）：" % C_SPARSE)
print("%8s %6s %12s %12s %9s" % ("L", "u", "全注意力", "ProbSparse", "省下"))
RATIOS = []
for l in (72, 168, 336, 720, 1440):
    u = max(1, int(math.ceil(C_SPARSE * math.log(l))))
    RATIOS.append((l, u, 1 - 2 * l * u / (l * l)))
    print("%8d %6d %12d %12d %8.1f%%" % (l, u, l * l, 2 * l * u, 100 * RATIOS[-1][2]))

LENS = [WINDOW, WINDOW // 2, WINDOW // 4]
print()
print("蒸馏（stride=2 的卷积）后序列长度：%s（总压缩 %d 倍）" % (LENS, LENS[0] // LENS[-1]))

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot([r[0] for r in RATIOS], [100 * r[2] for r in RATIOS], marker="o", color="tab:red", label="ProbSparse 省下的比例")
ax.plot([r[0] for r in RATIOS], [100 * r[1] / r[0] for r in RATIOS], marker="s", color="tab:blue",
        label="u/L（被选中的 query 占比）")
ax.set_xscale("log")
ax.set_xlabel("序列长度 L（对数刻度）")
ax.set_ylabel("百分比 %")
ax.set_title("ProbSparse 的收益随序列变长而变大")
ax.legend(fontsize=9)
plt.tight_layout()

# ---- 验收 ----
assert U72 == 22, "⌈5·ln72⌉ = ⌈21.38⌉ = 22"
assert score_full == 5184 and score_sparse == 3168
assert abs(score_sparse / score_full - 0.6111) < 1e-3, "L=72 时只省 38.9%"
assert RATIOS[-1][2] > 0.9, "L=1440 时省下 90% 以上"
assert LENS == [72, 36, 18]
assert max(1, int(math.ceil(C_SPARSE * math.log(720)))) == 33

## 任务 8：Transformer 训练，全注意力 vs ProbSparse（§8.5.1）

In [ ]:
D_MODEL, TF_EPOCHS, TF_BS, TF_LR = 32, 20, 256, 5e-3


class FullAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q, self.k, self.v = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.d = d_model

    def forward(self, x):
        Q, K, V = self.q(x), self.k(x), self.v(x)
        return torch.softmax(Q @ K.transpose(1, 2) / math.sqrt(self.d), -1) @ V


class TSFormer(nn.Module):
    """最小版时序 Transformer：正弦位置编码 + 1 层注意力 + 前馈，最后取最后一个时刻。"""

    def __init__(self, d_model=D_MODEL, attn="sparse", out=HORIZON):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        pe = torch.zeros(WINDOW, d_model)
        pos = torch.arange(WINDOW).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
        self.attn = FullAttention(d_model) if attn == "full" else ProbSparseAttention(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 2 * d_model), nn.ReLU(), nn.Linear(2 * d_model, d_model))
        self.head = nn.Linear(d_model, out)
        self.mode = attn

    def forward(self, x):
        h = self.embed(x.transpose(1, 2)) + self.pe.unsqueeze(0)
        h = self.norm(h + self.attn(h))
        h = self.norm(h + self.ff(h))
        return self.head(h[:, -1])


TF_PRED, TF_TIME = {}, {}
print("两个变体都用**全部** %d 个训练样本（Transformer 比 TCN 便宜得多，见 §8.4.4）" % len(Xt))
for mode in ("full", "sparse"):
    t0 = time.time()
    with torch.random.fork_rng():
        torch.manual_seed(3)                 # 同一个种子 → 两个变体**初始权重完全相同**
        model = TSFormer(attn=mode)
        opt = torch.optim.Adam(model.parameters(), lr=TF_LR)
        for ep in range(TF_EPOCHS):
            perm = torch.randperm(len(Xt))
            for i in range(0, len(Xt), TF_BS):
                idx = perm[i:i + TF_BS]
                opt.zero_grad()
                ((model(Xt[idx]) - Yt[idx]) ** 2).mean().backward()
                opt.step()
        model.eval()
        with torch.no_grad():
            TF_PRED[mode] = model(Xe).numpy() * SY + MY
    TF_TIME[mode] = time.time() - t0
    print("Transformer[%-6s] %5.1f s | 24 步 MAE %.6f（h=1 %.4f → h=24 %.4f）"
          % (mode, TF_TIME[mode], mae_all(Y_test, TF_PRED[mode]),
             mae_by_step(Y_test, TF_PRED[mode])[0], mae_by_step(Y_test, TF_PRED[mode])[23]))
print()
print("算力账：TCN 花了 %.1f s 只买到 %.1f 万次「样本-轮」；"
      % (tcn_secs, len(Xs) * EPOCHS / 1e4))
print("        Transformer 花 %.1f s 买到 %.1f 万次 —— 每个样本-轮便宜 %.1f 倍"
      % (TF_TIME["sparse"], len(Xt) * TF_EPOCHS / 1e4,
         (tcn_secs / (len(Xs) * EPOCHS)) / (TF_TIME["sparse"] / (len(Xt) * TF_EPOCHS))))
print("ProbSparse 与全注意力的 MAE 差 %+.4f | 训练耗时差 %+.1f s"
      % (mae_all(Y_test, TF_PRED["sparse"]) - mae_all(Y_test, TF_PRED["full"]),
         TF_TIME["sparse"] - TF_TIME["full"]))
print("注意：L=72 时 ProbSparse 只省 38.9%% 的打分次数，却多付了随机采样与 top-k 的开销，")
print("      所以在这个长度上「省时间」体现不出来 —— 收益要等 L 上百甚至上千（§8.5.1）")

# ---- 验收 ----
for mode in ("full", "sparse"):
    assert TF_PRED[mode].shape == (1714, HORIZON)
    assert 10 < mae_all(Y_test, TF_PRED[mode]) < 45, "随机训练结果 → 区间断言"
assert abs(mae_all(Y_test, TF_PRED["sparse"]) - mae_all(Y_test, TF_PRED["full"])) < 8, "两个变体应在同一量级"
assert mae_all(Y_test, TF_PRED["full"]) < mae_all(Y_test, P_tcn), "Transformer 比 TCN 准（TCN 受算力限制只训了 1/4 样本）"
assert U72 == 22

## 任务 9：全基线对比表（§8.6）

In [ ]:
preds = {
    "last value": P_last,
    "seasonal naive t-24": P_sn24,
    "seasonal naive t-168": P_sn168,
    "moving average 24": P_ma24,
    "LightGBM（20 特征）": P_lgb,
    "TCN（因果卷积 + 膨胀）": P_tcn,
    "Transformer（全注意力）": TF_PRED["full"],
    "Transformer（ProbSparse）": TF_PRED["sparse"],
}
ranked = sorted(((nm, mae_all(Y_test, P)) for nm, P in preds.items()), key=lambda kv: kv[1])

base = dict(ranked)["seasonal naive t-168"]
print("%-26s %10s %11s %10s" % ("模型 / 基线", "24步 MAE", "相对基线", "h=1 MAE"))
for nm, v in ranked:
    print("%-26s %10.4f %10.1f%% %10.4f"
          % (nm, v, 100 * v / base, mae_by_step(Y_test, preds[nm])[0]))
print()
print("及格线（最好的朴素基线）= %.6f" % base)
print("第一名 %s = %.6f，相对基线提升 %.1f%%"
      % (ranked[0][0], ranked[0][1], 100 * (1 - ranked[0][1] / base)))

names = [nm for nm, _ in ranked]
vals = [v for _, v in ranked]
colors = ["tab:red" if nm.startswith("LightGBM") else
          ("tab:gray" if any(k in nm for k in ("last", "seasonal", "moving")) else "tab:blue")
          for nm in names]
fig, ax = plt.subplots(figsize=(9.4, 4.4))
ax.barh(names[::-1], vals[::-1], color=colors[::-1])
ax.axvline(base, color="black", ls="--", lw=1.2, label="及格线（朴素基线 %.1f）" % base)
ax.set_xlabel("24 步 MAE (kW)")
ax.set_title("全基线对比：特征工程 + LightGBM 第一，深度模型没打过它")
ax.legend(fontsize=9)
plt.tight_layout()

# ---- 验收 ----
assert len(ranked) == 8 and len(preds) == 8
assert [v for _, v in ranked] == sorted(vals), "必须真的按 MAE 升序"
assert ranked[0][0] == "LightGBM（20 特征）", "本章结论：LightGBM 排第一"
assert ranked[-1][0] == "last value" and ranked[-1][1] == max(vals)
assert dict(ranked)["seasonal naive t-168"] < dict(ranked)["seasonal naive t-24"], "周朴素强于日朴素"
assert ranked[0][1] / base < 0.6
assert dict(ranked)["TCN（因果卷积 + 膨胀）"] > ranked[0][1], "深度模型输给了 LightGBM"

## 自查清单（能不看笔记答出来才算过）

- [ ] 9 个代码块的 assert 全部通过
- [ ] 四个朴素基线的 24 步 MAE 分别是多少？及格线是谁？
- [ ] 为什么 `t-168` 比 `t-24` 还好？它对齐了哪几个周期因子？
- [ ] `last value` 的整体 MAE 最差，为什么它的 h=24 反而最小？
- [ ] 树模型为什么也要求日历特征做 sin·cos 编码？23 点与 0 点的编码距离是多少？
- [ ] LightGBM 与 sklearn HistGB 的三个主要差异是什么（分裂方式 / 正则 / 早停）？
- [ ] 因果卷积为什么只补左边？补多少格？
- [ ] 感受野公式写出来，并代入本章参数算出结果；实测值是多少？
- [ ] RF=61 < 窗口 72 意味着什么？TCN 为什么打不过 LightGBM？
- [ ] ProbSparse 的 `u` 怎么算？L=72 时 u 是多少、省下多少打分次数？
- [ ] 为什么 L=72 时 ProbSparse 并不更快？
- [ ] 蒸馏把 72 变成多少？它砍掉的是什么？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 样本数 / 切分点 / 训练 / 测试 | 8569 / 6855 / 6855 / 1714 |
| 测试段时段 | 2025-10-20 15:00 ~ 2025-12-31 00:00 |
| last value：24 步 / h1 / h24 MAE | 60.524392 / 30.0939 / 21.6066 |
| seasonal naive t-24 | 32.998794（h1 33.0215、h24 33.1928） |
| **seasonal naive t-168（及格线）** | **31.560227**（h1 31.5845、h24 31.6804） |
| moving average 24 | 45.527850 |
| 特征个数 / 拟合段 / 验证段 / 测试段 | 20 / 5826 / 1029 / 1714 |
| 感受野公式值 = 实测值 | 61 |
| 被浪费的输入格数（72 − 61） | 11 |
| 因果卷积层数（含输入层） | 9 |
| ProbSparse：u（L=72, c=5） | 22 |
| 打分次数：全注意力 / ProbSparse | 5184 / 3168（省 38.9%） |
| L=720 时省下的打分比例 | 90.8%（u=33） |
| L=1440 时省下的打分比例 | 94.9%（u=37） |
| 蒸馏后长度链 | 72 → 36 → 18 |
| TCN 训练样本数（stride=4）/ Transformer 训练样本数 | 1714 / 6855 |

> 上面全是**确定性**的量（不同机器上重跑一模一样），所以代码里可以卡死值。
> 下面是**训练出来的**量，只做区间断言，这张表里给的是本章实跑的一次参考值：

| 量（参考值，随种子/线程数浮动） | 值 |
|---|---|
| LightGBM 24 步 / h1 / h24 MAE | 15.720011 / 15.0203 / 16.0834 |
| HistGB 三点平均 vs LightGBM 三点平均 | 15.7334 vs 15.4988（差 +0.2346） |
| TCN 24 步 / h1 / h24 MAE（stride=4，12 轮） | 29.980845 / 27.5360 / 22.2883 |
| Transformer 全注意力 / ProbSparse 24 步 MAE | 20.033704 / 20.200566 |
| Transformer 两个变体的训练耗时 | 12.6 s / 19.6 s |